Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## Memory

- A model has no memory between calls ; each one starts from nothing
- "Memory" is the earlier conversation replayed into the next prompt
- `session_id` is what keeps two conversations from mixing

Wraps a chain in message history, then asks a follow-up that only works if the
first exchange was remembered.

### Exercise RunnableWithMessageHistory
Build a simple chain with conversation memory.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = make_llm()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant. Remember the conversation context."),
    MessagesPlaceholder("history"),
    ("user", "{input}"),
])

chain = prompt | llm

store = {}
def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history=get_history,
    input_messages_key="input",
    history_messages_key="history",
)

resp1 = chain_with_memory.invoke({"input": "Hi, my name is Pipo."}, config={"configurable": {"session_id": "user_session_1"}})
print(resp1.content)

resp2 = chain_with_memory.invoke({"input": "What time is my appointment?"}, config={"configurable": {"session_id": "user_session_1"}})
print(resp2.content)

### Solution 
The magic is the session_id. RunnableWithMessageHistory wraps the chain so that, keyed on that id, it pulls the past messages out of store, injects them into the MessagesPlaceholder("history") slot before calling the model, and afterwards appends both the new input and the model's reply back into that session's history. So the second call's prompt silently contains the first exchange, which is how a stateless model "remembers."

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# LLM on local Ollama : no .env or API key needed
llm = make_llm()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant. Remember the conversation context."),
    MessagesPlaceholder("history"),
    ("user", "{input}"),
])

chain = prompt | llm

store = {}
def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history=get_history,
    input_messages_key="input",
    history_messages_key="history",
)

# First turn: introduce your name
resp1 = chain_with_memory.invoke(
    {"input": "Hi, my name is Pipo."},
    config={"configurable": {"session_id": "user_session_1"}},
)
print(resp1.content)

# Second turn: same session_id → the chain should recall the name
resp2 = chain_with_memory.invoke(
    {"input": "What time is my appointment?"},
    config={"configurable": {"session_id": "user_session_1"}},
)
print(resp2.content)

### Exercise Conversation memory: add a conversation buffer to the chain.
Demonstrate integrating the history buffer with the chain.

#### Optional : write it first

The next cell is the finished version ; nothing below depends on doing this first.

```python
from langchain_core.chat_history import InMemoryChatMessageHistory

memory = InMemoryChatMessageHistory()
memory.add_user_message("Hi!")
memory.add_ai_message("Hi, how can I help?")
____  # <- add another message to the history

for ____ in memory.messages:   # complete the loop and display the saved messages
    print(____)
```

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory

memory = InMemoryChatMessageHistory()
memory.add_user_message("Hi!")
memory.add_ai_message("Hi, how can I help?")
memory.add_user_message("How do I reschedule an appointment?")
memory.add_ai_message("A framework for building applications with LLMs.")

for message in memory.messages:
    print(f"{message.type}: {message.content}")

### Try a second session_id

- Run the follow-up question again under a different `session_id`
- It should fail to understand, because that conversation never happened
- Every remembered turn is re-sent on the next call, so memory costs tokens